# Lilly v2 — step 7: PP-OCRv6 recogniser fine-tune (`lilly-ocr-paddle`)

Pre-registered before this notebook existed: `training/PREREGISTRATION.md`,
"step 7, PP-OCRv6 recogniser fine-tune". Everything fixed there is fixed here:
data (`lilly-ocr-crops`, Latin, ≤ 25 chars, split by source photograph),
recipe (PaddleOCR v3.7.0 `PP-OCRv6_medium_rec.yml`, 30 epochs, lr 1e-4,
warmup 2, Baidu's training checkpoint), model selection (`best_accuracy` on
the validation crops only), and the crop gate (fine-tuned ≥ stock exact on the
validation crops, one process, or `SystemExit` and no zip).

**Required attach:** dataset `lilly-ocr-crops` (`crops.zip`: `crops/*.png` at
the zip root, `crops2/*.png` under `crops2/`). Missing → `SystemExit`.

**Not done here:** the photograph bars. test-v2 is never on this box. The
exported model is scored through the app's door (`LILLY_PADDLE_REC_DIR`) at
the shipped floor 0.9 on the cloud or the Mac, one run, per the pre-registration.

Not EasyOCR: none of the old trainer, the old reader's weights, plates or the
closed street-photograph line.
Rules: `docs/kaggle-notebooks.md` — `run()` tees to `stdout.txt`, every
failure raises, ERROR means do not install.


In [ ]:
# 1. Stop here unless the machine is actually set up
import json, os, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

import torch  # the image's CUDA build, only to assert a GPU is attached
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

PRETRAINED_URL = ("https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/"
                  "PP-OCRv6_medium_rec_pretrained.pdparams")
for host in ("https://github.com", "https://pypi.org", "https://huggingface.co", PRETRAINED_URL):
    reachable(host)
print("network ok")

TEE = Path("/kaggle/working/stdout.txt")

def run(*cmd, quiet=False, env=None):
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1, env=env)
        for out in child.stdout:
            if not quiet:
                print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)


In [ ]:
# 2. Get the Lilly code — into scratch, NOT /kaggle/working (Output holds only the zips)
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training" / "paddle_rec_data.py").is_file(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd())
from training.kaggle_offload import Offload
OFF = Offload("ocr-paddle", os.environ.get("LILLY_RUN_ID", "ocr-paddle"))
OFF.hardware(torch.cuda.get_device_name(0))
# Offload writes experiment_log.json, metrics.jsonl and artifacts_manifest.json under
# /kaggle/working; OFF.check_trainproof() scans the tee before anything is zipped.


In [ ]:
# 3. PaddlePaddle with CUDA, PaddleOCR at the pinned version, and its training code at the same tag
PADDLEOCR_TAG = "v3.7.0"     # matches requirements.txt: paddleocr==3.7.0
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line and not line.startswith("--"):
        pins[line.split("==")[0].strip().lower()] = line
assert pins["paddleocr"] == "paddleocr==3.7.0", pins.get("paddleocr")
paddle_version = pins["paddlepaddle"].split("==")[1]
# Do NOT pip-install torch. The CPU paddlepaddle pin is for the Mac; here the GPU wheel of the same version.
run(sys.executable, "-m", "pip", "install", "-q", f"paddlepaddle-gpu=={paddle_version}",
    "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/")
run(sys.executable, "-m", "pip", "install", "-q", pins["paddleocr"], pins.get("opencv-contrib-python", "opencv-contrib-python"))
POCR = SCRATCH / "PaddleOCR"
subprocess.run(["rm", "-rf", str(POCR)], check=True)
run("git", "clone", "-q", "--depth", "1", "--branch", PADDLEOCR_TAG, "https://github.com/PaddlePaddle/PaddleOCR.git", str(POCR))
run(sys.executable, "-m", "pip", "install", "-q", "-r", str(POCR / "requirements.txt"))
CFG = POCR / "configs" / "rec" / "PP-OCRv6" / "PP-OCRv6_medium_rec.yml"
assert CFG.is_file(), f"{CFG} missing at {PADDLEOCR_TAG}"
import paddle
assert paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0, (
    "paddlepaddle-gpu landed without CUDA — the wheel index or the CUDA version is wrong; fix, do not train on CPU")
print("paddle", paddle.__version__, "on", paddle.device.get_device())


In [ ]:
# 4. Smoke: the GPU computes, the pretrained checkpoint loads (seconds, not hours)
x = paddle.rand([256, 256]); y = (x @ x).sum(); y.backward(); print("cuda matmul ok:", float(y) > 0)
PRETRAINED = SCRATCH / "PP-OCRv6_medium_rec_pretrained.pdparams"
run("curl", "-sSfL", "-o", str(PRETRAINED), PRETRAINED_URL)
assert PRETRAINED.stat().st_size > 10_000_000, "pretrained checkpoint too small — download failed"
state = paddle.load(str(PRETRAINED))
print("pretrained tensors:", len(state))
assert len(state) > 100, "the checkpoint did not load as a state dict"
del state


In [ ]:
# 5. Required data: the blind-labelled crops (lilly-ocr-crops) → PaddleOCR lists, split by source photograph
zips = sorted(Path("/kaggle/input").rglob("crops.zip"))
if not zips:
    raise SystemExit("lilly-ocr-crops is not attached (no crops.zip under /kaggle/input). "
                     "Relaunch with: python3 scripts/kaggle_train.py ocr-paddle")
DATA = SCRATCH / "train_data"
subprocess.run(["rm", "-rf", str(DATA)], check=True)
(DATA / "crops").mkdir(parents=True); (DATA / "crops2").mkdir(parents=True)
run("unzip", "-q", "-o", str(zips[0]), "-d", str(DATA / "crops"))
# crops2 is packed under a crops2/ prefix inside the same zip
if (DATA / "crops" / "crops2").is_dir():
    run("bash", "-c", f"mv {DATA / 'crops' / 'crops2'}/* {DATA / 'crops2'}/ && rmdir {DATA / 'crops' / 'crops2'}")
for sub in ("crops", "crops2"):
    src = CLONE / "data" / "ocr" / sub / "labels-human.tsv"
    assert src.is_file(), f"{src} missing in the clone"
    (DATA / sub / "labels-human.tsv").write_text(src.read_text(encoding="utf-8"), encoding="utf-8")
n_png = len(list((DATA / "crops").glob("*.png"))) + len(list((DATA / "crops2").glob("*.png")))
print("crops on disk:", n_png)
assert n_png >= 1500, f"only {n_png} PNGs — the dataset did not land"
run(sys.executable, "training/paddle_rec_data.py", "--root", str(DATA))   # refuses a missing PNG or a leaking split
report = json.loads((DATA / "split-report.json").read_text(encoding="utf-8"))
assert report["train"]["crops"] >= 1000 and report["valid"]["crops"] >= 300, report
OFF.metric("train_crops", report["train"]["crops"]); OFF.metric("valid_crops", report["valid"]["crops"])


In [ ]:
# 6. Train — the pre-registered recipe, nothing else
OUT = SCRATCH / "output" / "rec"
subprocess.run(["rm", "-rf", str(OUT)], check=True)
os.chdir(POCR)
run(sys.executable, "tools/train.py", "-c", str(CFG), "-o",
    f"Global.pretrained_model={PRETRAINED}",
    "Global.epoch_num=30",
    "Optimizer.lr.learning_rate=0.0001",
    "Optimizer.lr.warmup_epoch=2",
    "Global.eval_batch_step=[0,18]",
    "Global.save_epoch_step=5",
    "Global.cal_metric_during_train=true",
    "Global.use_gpu=true",
    f"Global.save_model_dir={OUT}",
    f"Train.dataset.data_dir={DATA}",
    f"Train.dataset.label_file_list=[{DATA / 'train_list.txt'}]",
    f"Eval.dataset.data_dir={DATA}",
    f"Eval.dataset.label_file_list=[{DATA / 'val_list.txt'}]")
os.chdir(CLONE)
tee = TEE.read_text(encoding="utf-8", errors="replace")
train_lines = [l for l in tee.splitlines() if "loss:" in l and "epoch:" in l]
assert train_lines, "no training loss line reached the tee — the trainer never ran"
assert not any(("nan" in l.lower() or "inf" in l.lower()) and "loss:" in l for l in train_lines), "NaN/Inf loss — refused"
assert any("epoch: [30/30]" in l for l in train_lines), "the run never reached epoch 30 — refused"
best = OUT / "best_accuracy.pdparams"
assert best.is_file(), "no best_accuracy checkpoint — the trainer never evaluated"
print("training lines:", len(train_lines), "| last:", train_lines[-1][-160:])


In [ ]:
# 7. Export the best checkpoint to inference format (what the app loads through LILLY_PADDLE_REC_DIR)
EXPORT = SCRATCH / "export" / "rec"
subprocess.run(["rm", "-rf", str(EXPORT)], check=True)
os.chdir(POCR)
run(sys.executable, "tools/export_model.py", "-c", str(CFG), "-o",
    f"Global.pretrained_model={OUT / 'best_accuracy'}", f"Global.save_inference_dir={EXPORT}")
os.chdir(CLONE)
for f in ("inference.pdiparams", "inference.yml"):
    assert (EXPORT / f).is_file(), f"export missing {f}"
assert (EXPORT / "inference.json").is_file() or (EXPORT / "inference.pdmodel").is_file(), "export missing the model graph"
print(sorted(p.name for p in EXPORT.iterdir()))


In [ ]:
# 8. Crop gate — stock and fine-tuned in ONE process on the validation crops, before any zip
import random, unicodedata
from PIL import Image
import numpy as np
from paddleocr import TextRecognition

def fold(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)).casefold()

val = [l.rstrip("\n").split("\t") for l in (DATA / "val_list.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
imgs = [np.asarray(Image.open(DATA / p).convert("RGB"))[:, :, ::-1] for p, _ in val]   # BGR, as the pipeline feeds it
def score(model):
    out = []
    for i in range(0, len(imgs), 64):
        out += [str(r["rec_text"]).strip() for r in model.predict(imgs[i:i + 64])]
    return out
stock = score(TextRecognition(model_name="PP-OCRv6_medium_rec", enable_mkldnn=False))
tuned = score(TextRecognition(model_name="PP-OCRv6_medium_rec", model_dir=str(EXPORT), enable_mkldnn=False))
assert len(stock) == len(tuned) == len(val)
def acc(pred, folded=False):
    return sum((fold(p) == fold(t)) if folded else (p == t) for p, (_, t) in zip(pred, val))
n = len(val)
s_ex, t_ex, s_fo, t_fo = acc(stock), acc(tuned), acc(stock, True), acc(tuned, True)
deltas = [int(p2 == t) - int(p1 == t) for p1, p2, (_, t) in zip(stock, tuned, val)]
rng = random.Random(0); means = sorted(sum(rng.choice(deltas) for _ in range(n)) / n for _ in range(5000))
lo, hi = 100 * means[125], 100 * means[4874]
gate = {"n": n, "stock_exact": s_ex, "tuned_exact": t_ex, "stock_folded": s_fo, "tuned_folded": t_fo,
        "delta_exact_points": 100 * (t_ex - s_ex) / n, "ci95": [lo, hi]}
Path("/kaggle/working/crop-gate.json").write_text(json.dumps(gate, indent=1))
print(f"validation crops n={n}: stock exact {s_ex} ({100*s_ex/n:.1f}%) folded {s_fo}; "
      f"fine-tuned exact {t_ex} ({100*t_ex/n:.1f}%) folded {t_fo}; Δ exact {100*(t_ex-s_ex)/n:+.1f} points (95% {lo:+.1f} to {hi:+.1f})")
for k, v in gate.items():
    OFF.metric(f"crop_gate_{k}", v)
if t_ex < s_ex:
    OFF.fail(f"crop gate refused: fine-tuned exact {t_ex} < stock {s_ex} on {n} validation crops")
    raise SystemExit("crop gate refused — the fine-tune reads the held-out crops worse than the stock model. No zip.")
print("crop gate passed (it ships nothing; test-v2 decides)")


In [ ]:
# 9. Zip the exported recogniser only — after the gate, nothing else in Output
OFF.check_trainproof()
ZIP = Path("/kaggle/working/lilly-read-paddle.zip")
run("bash", "-c", f"cd {EXPORT.parent} && zip -qr {ZIP} {EXPORT.name}")
assert ZIP.stat().st_size > 1_000_000, "zip too small"
OFF.finish("complete", [str(ZIP), "/kaggle/working/crop-gate.json"])
print("wrote", ZIP, ZIP.stat().st_size // 1_000_000, "MB — score it on test-v2 through LILLY_PADDLE_REC_DIR before believing anything")


**ERROR or CANCEL means do not install.** A zip left over from an earlier
version is not this run's product. Fix the cause, push, relaunch. The crop gate
only says the fine-tune did not get worse on held-out crops; the two test-v2
bars (per photograph rise with the 95% interval excluding zero, invented ≤ 450)
are measured off this box and decide alone.
